## Étape 4.1 : lecture des données

Le CSV est lu depuis HDFS. On vérifie le nombre de colonnes (24 attendues) et de lignes (environ 200 000 attendues).

In [1]:
import os
from pyspark.sql import SparkSession

DATA_PATH = os.environ.get(
    "DATA_PATH",
    "hdfs://localhost:9000/user/miguel/projet_3IABD/Bank_Transaction_Fraud_Detection.csv",
)

OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

spark = SparkSession.builder.appName("fraude").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Lecture du CSV depuis HDFS.
raw = spark.read.csv(DATA_PATH, header=True, inferSchema=False)

# Contrôles : 24 colonnes attendues, environ 200 000 lignes attendues
print("Nombre de colonnes :", len(raw.columns))
print("Nombre de lignes :", raw.count())

raw.printSchema()

26/11/19 10:37:53 WARN Utils: Your hostname, LAPTOP-2O6PEQV6 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/19 10:37:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 10:37:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Nombre de colonnes : 24


Nombre de lignes : 200000
root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Bank_Branch: string (nullable = true)
 |-- Account_Type: string (nullable = true)
 |-- Transaction_ID: string (nullable = true)
 |-- Transaction_Date: string (nullable = true)
 |-- Transaction_Time: string (nullable = true)
 |-- Transaction_Amount: string (nullable = true)
 |-- Merchant_ID: string (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Merchant_Category: string (nullable = true)
 |-- Account_Balance: string (nullable = true)
 |-- Transaction_Device: string (nullable = true)
 |-- Transaction_Location: string (nullable = true)
 |-- Device_Type: string (nullable = true)
 |-- Is_Fraud: string (nullable = true)
 |-- Transaction_Currency: string (nullable = true)
 |-- Customer_Contact: string (n

## Étape 4.2 : renommage et typage des colonnes

Les colonnes sont renommées en français, puis typées : les montants, l'âge et la cible en nombres, la date en type date, le reste en texte.

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

# Correspondance entre les noms
RENOMMAGE = {
    "Customer_ID": "id_client",
    "Customer_Name": "nom_client",
    "Gender": "genre",
    "Age": "age",
    "State": "etat",
    "City": "ville",
    "Bank_Branch": "agence_bancaire",
    "Account_Type": "type_compte",
    "Transaction_ID": "id_transaction",
    "Transaction_Date": "date_transaction",
    "Transaction_Time": "heure_transaction",
    "Transaction_Amount": "montant",
    "Merchant_ID": "id_marchand",
    "Transaction_Type": "type_transaction",
    "Merchant_Category": "categorie_marchand",
    "Account_Balance": "solde_compte",
    "Transaction_Device": "appareil_transaction",
    "Transaction_Location": "localisation_transaction",
    "Device_Type": "type_appareil",
    "Is_Fraud": "est_fraude",
    "Transaction_Currency": "devise",
    "Customer_Contact": "contact_client",
    "Transaction_Description": "description_transaction",
    "Customer_Email": "email_client",
}

# Contrôle : toutes les colonnes attendues doivent exister dans le fichier.
manquantes = [c for c in RENOMMAGE if c not in raw.columns]
if manquantes:
    raise ValueError(f"Colonnes absentes du fichier : {manquantes}")


df = raw.select([F.col(c).alias(RENOMMAGE[c]) for c in RENOMMAGE])

# Typage des colonnes
df = (
    df
    .withColumn("age", F.col("age").cast(IntegerType()))
    .withColumn("est_fraude", F.col("est_fraude").cast(IntegerType()))
    .withColumn("montant", F.col("montant").cast(DoubleType()))
    .withColumn("solde_compte", F.col("solde_compte").cast(DoubleType()))
    .withColumn("date_transaction", F.to_date(F.col("date_transaction"), "dd-MM-yyyy"))
    .withColumn("heure_transaction", F.substring(F.col("heure_transaction"), 1, 2).cast(IntegerType()))
)

# Contrôle : une valeur qui ne se convertit pas devient null.
for c in ["age", "est_fraude", "montant", "solde_compte", "date_transaction", "heure_transaction"]:
    n = df.filter(F.col(c).isNull()).count()
    print(f"{c} : {n} valeur(s) non convertie(s)")

df.printSchema()

age : 0 valeur(s) non convertie(s)
est_fraude : 0 valeur(s) non convertie(s)
montant : 0 valeur(s) non convertie(s)
solde_compte : 0 valeur(s) non convertie(s)


date_transaction : 0 valeur(s) non convertie(s)
heure_transaction : 0 valeur(s) non convertie(s)
root
 |-- id_client: string (nullable = true)
 |-- nom_client: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- etat: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- agence_bancaire: string (nullable = true)
 |-- type_compte: string (nullable = true)
 |-- id_transaction: string (nullable = true)
 |-- date_transaction: date (nullable = true)
 |-- heure_transaction: integer (nullable = true)
 |-- montant: double (nullable = true)
 |-- id_marchand: string (nullable = true)
 |-- type_transaction: string (nullable = true)
 |-- categorie_marchand: string (nullable = true)
 |-- solde_compte: double (nullable = true)
 |-- appareil_transaction: string (nullable = true)
 |-- localisation_transaction: string (nullable = true)
 |-- type_appareil: string (nullable = true)
 |-- est_fraude: integer (nullable = true)
 |-- devise: stri

## Étape 4.3 : statistiques descriptives

On décrit les colonnes numériques, la répartition de la cible `est_fraude`, puis les liens entre les variables et la fraude. Lors d'une analyse préalable sur le fichier complet, le taux de fraude était d'environ 5 % et les corrélations proches de 0. Ces chiffres devront être retrouvés ici, et interprétés après exécution.

In [ ]:
from pyspark.sql import functions as F

colonnes_num = ["age", "montant", "solde_compte", "est_fraude"]

df.select(colonnes_num).describe().show()

# 3. Corrélation de chaque variable numérique avec la cible.
for c in ["age", "montant", "solde_compte"]:
    r = df.stat.corr(c, "est_fraude")
    print(f"corrélation {c} / est_fraude : {r:.4f}")

# 1. Nombre de transactions normales et frauduleuses
total = df.count()
(
    df.groupBy("est_fraude").count()
    .withColumn("taux_%", F.round(F.col("count") / total * 100, 2))
    .orderBy("est_fraude")
    .show()
)

# 2.Montant moyen par classe

df.groupBy("est_fraude").agg(F.round(F.avg("montant"), 2).alias("montant_moyen")).show()


#4. Taux de fraude par catégorie.
for c in ["type_transaction", "categorie_marchand", "type_appareil"]:
    (
        df.groupBy(c)
        .agg(F.round(F.avg("est_fraude") * 100, 2).alias("taux_fraude_%"))
        .orderBy(F.desc("taux_fraude_%"))
        .show()
    )

# 5. Export des statistiques descriptives
stats_pd = df.select(colonnes_num).describe().toPandas()
stats_pd.to_csv(f"{OUT}/statistiques.csv", index=False)
print("Statistiques exportées dans", f"{OUT}/statistiques.csv")

+-------+-----------------+-----------------+------------------+-------------------+
|summary|              age|          montant|      solde_compte|         est_fraude|
+-------+-----------------+-----------------+------------------+-------------------+
|  count|           200000|           200000|            200000|             200000|
|   mean|         44.01511|49538.01555420002|52437.988783549954|            0.05044|
| stddev|15.28877432696951|28551.87400382294| 27399.50712806853|0.21885165267877105|
|    min|               18|            10.29|           5000.82|                  0|
|    max|               70|         98999.98|          99999.95|                  1|
+-------+-----------------+-----------------+------------------+-------------------+



corrélation age / est_fraude : -0.0015


corrélation montant / est_fraude : -0.0021


corrélation solde_compte / est_fraude : 0.0001


+----------+------+------+
|est_fraude| count|taux_%|
+----------+------+------+
|         0|189912| 94.96|
|         1| 10088|  5.04|
+----------+------+------+



+----------+-------------+
|est_fraude|montant_moyen|
+----------+-------------+
|         1|     49277.93|
|         0|     49551.83|
+----------+-------------+



+----------------+-------------+
|type_transaction|taux_fraude_%|
+----------------+-------------+
|        Transfer|         5.19|
|          Credit|          5.1|
|           Debit|         5.08|
|    Bill Payment|         4.93|
|      Withdrawal|         4.93|
+----------------+-------------+



+------------------+-------------+
|categorie_marchand|taux_fraude_%|
+------------------+-------------+
|          Clothing|          5.2|
|         Groceries|         5.19|
|        Restaurant|         5.04|
|       Electronics|         5.03|
|            Health|         4.99|
|     Entertainment|         4.82|
+------------------+-------------+



+-------------+-------------+
|type_appareil|taux_fraude_%|
+-------------+-------------+
|      Desktop|          5.1|
|          POS|         5.05|
|          ATM|         5.03|
|       Mobile|         4.99|
+-------------+-------------+



Statistiques exportées dans outputs/statistiques.csv


## Étape 4.4 : nettoyage

Le nettoyage porte sur cinq points : suppression de la colonne constante `devise`, retrait des données personnelles, doublons sur l'identifiant de transaction, valeurs manquantes, et plages de valeurs. Lors de l'analyse préalable, aucun doublon ni valeur manquante n'a été trouvé : la cellule le vérifie et le documente.

In [4]:
from pyspark.sql import functions as F

# 1. Colonne constante
nb_devises = df.select("devise").distinct().count()
print("Valeurs distinctes de devise :", nb_devises)
df = df.drop("devise")

# 2. Doublons (AVANT de supprimer id_transaction)
n_avant = df.count()
df = df.dropDuplicates(["id_transaction"])
print("Doublons supprimés :", n_avant - df.count())

# 3. Données personnelles et identifiants uniques
df = df.drop("nom_client", "contact_client", "email_client", "id_client", "id_transaction", "id_marchand")

# 4. Valeurs manquantes
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(truncate=False)

# Suppression des lignes incomplètes
n_avant = df.count()
df = df.dropna()
print("Lignes incomplètes supprimées :", n_avant - df.count())

# 5. Retire les lignes incohérentes
n_avant = df.count()
df = df.filter(
    F.col("age").between(18, 120)
    & (F.col("montant") > 0)
    & (F.col("solde_compte") >= 0)
    & F.col("est_fraude").isin(0, 1)
)
print("Lignes incohérentes supprimées :", n_avant - df.count())

# Nombre de lignes et de colonnes conservées
print("Lignes après nettoyage :", df.count())
print("Colonnes après nettoyage :", len(df.columns))

Valeurs distinctes de devise : 1


Doublons supprimés : 0


+-----+---+----+-----+---------------+-----------+----------------+-----------------+-------+----------------+------------------+------------+--------------------+------------------------+-------------+----------+-----------------------+
|genre|age|etat|ville|agence_bancaire|type_compte|date_transaction|heure_transaction|montant|type_transaction|categorie_marchand|solde_compte|appareil_transaction|localisation_transaction|type_appareil|est_fraude|description_transaction|
+-----+---+----+-----+---------------+-----------+----------------+-----------------+-------+----------------+------------------+------------+--------------------+------------------------+-------------+----------+-----------------------+
|0    |0  |0   |0    |0              |0          |0               |0                |0      |0               |0                 |0           |0                   |0                       |0            |0         |0                      |
+-----+---+----+-----+---------------+----------

Lignes incomplètes supprimées : 0


Lignes incohérentes supprimées : 0


Lignes après nettoyage : 200000
Colonnes après nettoyage : 17


## Étape 4.5 : features

Les modèles ne lisent que des nombres. Cette étape extrait l'heure et le jour de la semaine, encode les colonnes texte, puis regroupe toutes les variables dans un vecteur `features`. Seules les colonnes texte à peu de valeurs distinctes sont retenues ; les colonnes à très nombreuses valeurs (ville, état, agence, localisation, appareil de transaction) sont écartées pour garder un modèle simple.

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

# 1. Feature engineering simple : extraction de l'heure et ratio montant / solde
df_feat = (
    df
    .withColumn("heure", F.substring(F.col("heure_transaction"), 1, 2).cast(IntegerType()))
    .withColumn("ratio_montant_solde", F.col("montant") / (F.col("solde_compte") + 1.0))
)

# 2. Gestion du déséquilibre (5% de fraude) via une colonne de poids
total = df_feat.count()
nb_fraudes = df_feat.filter(F.col("est_fraude") == 1).count()
nb_normales = total - nb_fraudes
poids_fraude = nb_normales / nb_fraudes  # ~19x plus de poids sur la classe fraude

df_feat = df_feat.withColumn(
    "poids",
    F.when(F.col("est_fraude") == 1, F.lit(poids_fraude)).otherwise(F.lit(1.0))
)

# 3. Séparation entraînement (80%) / test (20%) avant le Pipeline
train_raw, test_raw = df_feat.randomSplit([0.8, 0.2], seed=42)

# 4. Définition des variables utiles (les identifiants uniques sont exclus)
cols_cat = ["genre", "type_compte", "type_transaction", "categorie_marchand", "type_appareil"]
cols_num = ["age", "montant", "solde_compte", "heure", "ratio_montant_solde"]

# 5. Construction du Pipeline de prétraitement
indexeurs = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cols_cat]
encodeurs = [OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_ohe"]) for c in cols_cat]

entrees = [f"{c}_ohe" for c in cols_cat] + cols_num
assembleur = VectorAssembler(inputCols=entrees, outputCol="features_raw")
normalisateur = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)

pipeline = Pipeline(stages=indexeurs + encodeurs + [assembleur, normalisateur])

# Ajustement sur train uniquement, puis transformation sur train et test
modele_prep = pipeline.fit(train_raw)
train_df = modele_prep.transform(train_raw).select("features", "est_fraude", "poids")
test_df = modele_prep.transform(test_raw).select("features", "est_fraude", "poids")

print(f"Train : {train_df.count()} lignes | Test : {test_df.count()} lignes")

Train : 160365 lignes | Test : 39635 lignes


## Étape 4.6 : Machine Learning


.

In [6]:
import pandas as pd
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Définition des deux modèles avec prise en compte des poids
modeles = {
    "LogisticRegression": LogisticRegression(
        featuresCol="features",
        labelCol="est_fraude",
        weightCol="poids",
        maxIter=50
    ),
    "RandomForest": RandomForestClassifier(
        featuresCol="features",
        labelCol="est_fraude",
        weightCol="poids",
        numTrees=100,
        maxDepth=7,
        seed=42
    ),
}

# 2. Évaluateurs (AUPRC prioritaire pour les données déséquilibrées + AUROC en complément)
eval_pr = BinaryClassificationEvaluator(labelCol="est_fraude", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
eval_roc = BinaryClassificationEvaluator(labelCol="est_fraude", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
eval_f1_fraude = MulticlassClassificationEvaluator(
    labelCol="est_fraude",
    predictionCol="prediction",
    metricName="fMeasureByLabel",
    metricLabel=1.0  # Évalue uniquement la classe fraude (1)
)

resultats = []

# 3. Entraînement et évaluation de chaque modèle
for nom, algo in modeles.items():
    print(f"\n=== Entraînement : {nom} ===")
    modele = algo.fit(train_df)
    preds = modele.transform(test_df)

    auprc = eval_pr.evaluate(preds)
    auroc = eval_roc.evaluate(preds)
    f1_fraude = eval_f1_fraude.evaluate(preds)

    resultats.append({"modele": nom, "AUPRC": round(auprc, 4), "AUROC": round(auroc, 4), "F1_Fraude": round(f1_fraude, 4)})

    print(f"AUPRC : {auprc:.4f} | AUROC : {auroc:.4f} | F1_Fraude : {f1_fraude:.4f}")
    print("Matrice de confusion :")
    preds.groupBy("est_fraude", "prediction").count().orderBy("est_fraude", "prediction").show()

    

# 4. Tableau comparatif et export pour le dashboard (app.py)
res_df = pd.DataFrame(resultats)
print("\n=== Comparaison des modèles ===")
print(res_df.to_string(index=False))

res_df.to_csv(f"{OUT}/resultats_modeles.csv", index=False)
print(f"\nRésultats exportés dans {OUT}/resultats_modeles.csv")


=== Entraînement : LogisticRegression ===


26/11/19 10:38:50 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/11/19 10:38:50 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


AUPRC : 0.0524 | AUROC : 0.5029 | F1_Fraude : 0.0921
Matrice de confusion :


+----------+----------+-----+
|est_fraude|prediction|count|
+----------+----------+-----+
|         0|       0.0|21859|
|         0|       1.0|15711|
|         1|       0.0| 1207|
|         1|       1.0|  858|
+----------+----------+-----+


=== Entraînement : RandomForest ===


26/11/19 10:39:17 WARN DAGScheduler: Broadcasting large task binary with size 1031.8 KiB
26/11/19 10:39:20 WARN DAGScheduler: Broadcasting large task binary with size 1756.1 KiB
26/11/19 10:39:27 WARN DAGScheduler: Broadcasting large task binary with size 1551.4 KiB
26/11/19 10:39:30 WARN DAGScheduler: Broadcasting large task binary with size 1551.4 KiB
26/11/19 10:39:32 WARN DAGScheduler: Broadcasting large task binary with size 1563.0 KiB


AUPRC : 0.0528 | AUROC : 0.5072 | F1_Fraude : 0.0895
Matrice de confusion :


26/11/19 10:39:34 WARN DAGScheduler: Broadcasting large task binary with size 1565.3 KiB


+----------+----------+-----+
|est_fraude|prediction|count|
+----------+----------+-----+
|         0|       0.0|29511|
|         0|       1.0| 8059|
|         1|       0.0| 1591|
|         1|       1.0|  474|
+----------+----------+-----+


=== Comparaison des modèles ===
            modele  AUPRC  AUROC  F1_Fraude
LogisticRegression 0.0524 0.5029     0.0921
      RandomForest 0.0528 0.5072     0.0895

Résultats exportés dans outputs/resultats_modeles.csv


26/11/19 10:39:35 WARN DAGScheduler: Broadcasting large task binary with size 1535.4 KiB
